# Text-To-Voice Incident Response Assistant

Generates a short educational story on any topic with Mistral (via watsonx.ai), then reads it aloud with gTTS.

Set `WATSONX_API_KEY` and `WATSONX_PROJECT_ID` as environment variables before running.

In [ ]:
%pip install -q gTTS==2.5.4 ibm-watsonx-ai==1.1.20

## Model setup

In [ ]:
import os
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams

credentials = Credentials(
    url='https://us-south.ml.cloud.ibm.com',
    api_key=os.environ['WATSONX_API_KEY'],
)
project_id = os.environ['WATSONX_PROJECT_ID']

model = ModelInference(
    model_id='mistralai/mistral-medium-2505',
    credentials=credentials,
    project_id=project_id,
    params={
        GenParams.DECODING_METHOD: 'greedy',
        GenParams.MAX_NEW_TOKENS: 1000,
    },
)

## Threat Intel Report Generation

In [ ]:
def generate_cyber_story(scenario_topic, audience_level="Junior Analysts"):
    """
    Generates an engaging, scenario-based cybersecurity story
    explaining an attack chain and its defenses.
    """
    prompt = f"""You are a senior cybersecurity threat hunter and incident response storyteller.
Write an engaging, realistic narrative walkthrough about '{scenario_topic}' tailored for {audience_level}.

Structure the story as follows:
1. Scenario & Initial Access: Introduce the victim organization, the threat actor's initial entry vector, and the deception used.
2. Attack Execution: Describe lateral movement, privilege escalation, or payload delivery in clear technical steps without unnecessary jargon.
3. Detection & Containment: Show how defensive monitoring (SIEM/EDR alerts, log analysis) caught the anomalous behavior.
4. Lessons Learned: End with 3 concise, bulleted key takeaways covering mitigation, hardening, or detection rules.

Keep the total length between 250 and 350 words. Write in an engaging, narrative briefing tone suitable for audio playback."""

    response = model.generate_text(prompt=prompt)
    return response


# Test scenario
topic = "a phishing email leading to a multi-stage ransomware deployment via PsExec"
cyber_story = generate_cyber_story(topic, audience_level="SOC Analysts")

print("Generated Cybersecurity Briefing:\n")
print(cyber_story)

## Text to speech

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display
import io

def play_audio_briefing(text, lang='en', tld='co.uk'):
    """
    Converts briefing text to speech using gTTS and plays it in Jupyter.
    Using tld='co.uk' or 'com' provides different voice nuances.
    """
    # Clean up formatting characters that TTS might read awkwardly
    clean_text = text.replace('*', '').replace('#', '').replace('`', '')
    
    # Generate speech
    tts = gTTS(text=clean_text, lang=lang, tld=tld, slow=False)
    
    # Save to a memory buffer to avoid unnecessary disk writes
    fp = io.BytesIO()
    tts.write_to_fp(fp)
    fp.seek(0)
    
    # Display audio player
    display(Audio(fp.read(), autoplay=False))

# Play the briefing
play_audio_briefing(cyber_story)

In [ ]:
# Save the narration as an MP3
def export_briefing(text, filename="incident_briefing.mp3"):
    clean_text = text.replace('*', '').replace('#', '')
    tts = gTTS(text=clean_text, lang='en', tld='co.uk', slow=False)
    tts.save(filename)
    print(f"[+] Audio briefing saved successfully to {filename}")

export_briefing(cyber_story, "ransomware_case_study.mp3")